# Decision Trees

Adapted from material: https://www.kaggle.com/code/dansbecker/exercise-random-forests

Data has been downloaded from: https://www.kaggle.com/competitions/home-data-for-ml-course/data

In [29]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from utils.utils import fetch_dfs

Retrieve dfs and remove our target column from dfs.columns

In [3]:
df_train, df_test = fetch_dfs(
    'home-data-for-ml-course', 
    ['train.csv', 'test.csv']
)
target = 'SalePrice'
no_target = list(df_train.columns)
no_target.remove(target)

Separate input and output data to train

In [4]:
X = df_train[no_target].values
y = df_train[target].values

inputs = [
    'LotArea',
    'YearBuilt',
    '1stFlrSF',
    '2ndFlrSF',
    'FullBath',
    'BedroomAbvGr',
    'TotRmsAbvGrd'
]

xx = df_train[inputs]
yy = y.copy()

train_xx, test_xx, train_yy, test_yy = train_test_split(
    xx,
    yy,
    random_state=1
)

The RandomForestRegressor's mean absolute error is quite large

In [31]:
rfmodel = RandomForestRegressor(random_state=1)
rfmodel.fit(train_xx, train_yy)
rfpreds = rfmodel.predict(test_xx)
rfmae = mean_absolute_error(rfpreds, test_yy)
print(rfmae)

21857.15912981083


Removing columns with NaN values to see if it improves accuracy

In [24]:
floats = [
    col 
    for col in df_train.columns
    if df_train[col].dtype in ['float64', 'int64']
]
df_train[floats] \
    .isnull() \
    .sum()  \
    .sort_values(ascending=False) \
    .head(10)

LotFrontage     259
GarageYrBlt      81
MasVnrArea        8
LotArea           0
MSSubClass        0
Id                0
OverallCond       0
OverallQual       0
YearRemodAdd      0
YearBuilt         0
dtype: int64

In [25]:
df = df_train[floats].isnull().sum()
has_nans = df.loc[df > 0].index
for col in has_nans:
    floats.remove(col)

### Re-training with `floats`

In [27]:
new_X = df_train[floats]
X_train, X_test, y_train, y_test = train_test_split(
    new_X, y, 
    test_size=0.15,
    random_state=42
)

Looks better...

In [30]:
rfr = RandomForestRegressor(random_state=42)
rfr.fit(X_train, y_train)
y_pred = rfr.predict(X_test)
rfr_mae = mean_absolute_error(y_test, y_pred)
print(rfr_mae)

988.9164383561646
